In [ ]:
import os, json, time, zipfile
from datetime import datetime
import numpy as np
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt
from IPython.display import Markdown, Image, display

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

CARPETAS = ["redonda", "blanca", "negra", "corchea", "silencio_negra", "silencio_blanca",
            "clave_sol", "clave_fa", "sostenido", "bemol"]
NOMBRES = ["Redonda", "Blanca", "Negra", "Corchea", "Silencio de negra", "Silencio de blanca",
           "Clave de sol", "Clave de fa", "Sostenido", "Bemol"]
ABREV = ["Red", "Bla", "Neg", "Cor", "SNe", "SBl", "CSo", "CFa", "Sos", "Bem"]
NUM_CLASES = 10
TAM = 32
EPOCAS = 40
LOTE = 64
VAL = 0.15
PRUEBA = 0.15
AUTOR_PRUEBA = "test_test"
SEMILLA = 42
SUAVIZADO_ETIQUETAS = 0.1
SALIDA = "resultados"
os.makedirs(SALIDA, exist_ok=True)
keras.utils.set_random_seed(SEMILLA)

In [ ]:
for f in ["X.npy", "y.npy", "grupos.npy", "autores.npy"]:
    if os.path.exists(f):
        os.remove(f)
from google.colab import files
files.upload()
print(sorted(f for f in os.listdir(".") if f.endswith(".npy")))

In [ ]:
X = np.load("X.npy").astype("float32")[..., None] / 255.0
y = np.load("y.npy").astype("int64")
autores = np.load("autores.npy")
grupos = np.load("grupos.npy") if os.path.exists("grupos.npy") else np.arange(len(y))
print("Autores:", dict(zip(*np.unique(autores, return_counts=True))))
print("Imágenes:", X.shape, "| dibujos originales:", len(np.unique(grupos)))
print("Por clase:", dict(zip(NOMBRES, np.bincount(y, minlength=NUM_CLASES).tolist())))

In [ ]:
def dividir_por_grupos(y, grupos, val=VAL, prueba=PRUEBA, semilla=SEMILLA):
    rng = np.random.default_rng(semilla)
    sets = {"ent": [], "val": [], "pru": []}
    for c in np.unique(y):
        gs = rng.permutation(np.unique(grupos[y == c]))
        n_pru = max(1, round(len(gs) * prueba)) if prueba > 0 else 0
        n_val = max(1, round(len(gs) * val))
        corte = {"pru": gs[:n_pru], "val": gs[n_pru:n_pru + n_val], "ent": gs[n_pru + n_val:]}
        for k, g in corte.items():
            sets[k].append(np.where((y == c) & np.isin(grupos, g))[0])
    return tuple(np.sort(np.concatenate(sets[k])) for k in ("ent", "val", "pru"))


if AUTOR_PRUEBA:
    i_prueba = np.where(autores == AUTOR_PRUEBA)[0]
    resto = np.where(autores != AUTOR_PRUEBA)[0]
    a, b, _ = dividir_por_grupos(y[resto], grupos[resto], val=VAL / (1 - PRUEBA), prueba=0.0)
    i_ent, i_val = resto[a], resto[b]
    print(f"Prueba = todos los dibujos de '{AUTOR_PRUEBA}' (no vistos en el entrenamiento)")
else:
    i_ent, i_val, i_prueba = dividir_por_grupos(y, grupos)
Xe, ye = X[i_ent], y[i_ent]
Xv, yv = X[i_val], y[i_val]
Xp, yp = X[i_prueba], y[i_prueba]
conteos = {k: np.bincount(v, minlength=NUM_CLASES).tolist() for k, v in (("ent", ye), ("val", yv), ("pru", yp))}
print("Entrenamiento:", len(ye), "| Validación:", len(yv), "| Prueba:", len(yp))

In [ ]:
aumento = keras.Sequential([
    layers.RandomRotation(0.025, fill_mode="constant"),
    layers.RandomZoom((-0.12, 0.12), fill_mode="constant"),
    layers.RandomTranslation(0.08, 0.08, fill_mode="constant"),
    layers.GaussianNoise(0.04),
], name="aumento")


def crear_mlp():
    return keras.Sequential([
        keras.Input((TAM, TAM, 1)),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASES, activation="softmax"),
    ], name="mlp")


def crear_cnn():
    return keras.Sequential([
        keras.Input((TAM, TAM, 1)),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(2),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(NUM_CLASES, activation="softmax"),
    ], name="cnn")


def entrenar(modelo):
    modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
                   loss=keras.losses.CategoricalCrossentropy(label_smoothing=SUAVIZADO_ETIQUETAS),
                   metrics=[keras.metrics.CategoricalAccuracy(name="accuracy")])
    ds = (tf.data.Dataset.from_tensor_slices((Xe, ye))
          .shuffle(len(ye), seed=SEMILLA)
          .batch(LOTE)
          .map(lambda a, b: (tf.clip_by_value(aumento(a, training=True), 0.0, 1.0), tf.one_hot(b, NUM_CLASES)),
               num_parallel_calls=tf.data.AUTOTUNE)
          .prefetch(tf.data.AUTOTUNE))
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
    ]
    t0 = time.time()
    h = modelo.fit(ds, validation_data=(Xv, keras.utils.to_categorical(yv, NUM_CLASES)), epochs=EPOCAS, callbacks=callbacks, verbose=2, shuffle=False)
    return h.history, time.time() - t0


def evaluar(modelo, Xs, ys):
    t0 = time.time()
    probs = modelo.predict(Xs, verbose=0, batch_size=256)
    ms = 1000 * (time.time() - t0) / max(1, len(ys))
    pred = probs.argmax(1)
    p, r, f, s = precision_recall_fscore_support(ys, pred, labels=range(NUM_CLASES), zero_division=0)
    pm, rm, fm, _ = precision_recall_fscore_support(ys, pred, average="macro", zero_division=0)
    return {"exactitud": float(accuracy_score(ys, pred)), "precision_macro": float(pm),
            "recall_macro": float(rm), "f1_macro": float(fm),
            "por_clase": [{"clase": NOMBRES[i], "precision": float(p[i]), "recall": float(r[i]),
                           "f1": float(f[i]), "soporte": int(s[i])} for i in range(NUM_CLASES)],
            "matriz": confusion_matrix(ys, pred, labels=range(NUM_CLASES)).tolist(),
            "pred": pred, "probs": probs, "ms_por_imagen": ms}

In [ ]:
resultados, historiales, modelos = {}, {}, {}
for nombre, fabrica in (("mlp", crear_mlp), ("cnn", crear_cnn)):
    print(f"\n===== {nombre.upper()} =====")
    keras.utils.set_random_seed(SEMILLA)
    modelo = fabrica()
    modelo.summary()
    hist, seg = entrenar(modelo)
    ev = evaluar(modelo, Xp, yp)
    ev.update({"parametros": modelo.count_params(), "segundos": seg, "epocas": len(hist["loss"]),
               "val_exactitud": evaluar(modelo, Xv, yv)["exactitud"]})
    resultados[nombre], historiales[nombre], modelos[nombre] = ev, hist, modelo
    modelo.save(f"{SALIDA}/modelo_{nombre}.keras")
    print(f"{nombre.upper()}: exactitud prueba {ev['exactitud']:.4f} | F1 macro {ev['f1_macro']:.4f}")
mejor = max(("mlp", "cnn"), key=lambda n: resultados[n]["val_exactitud"])
print("\nModelo elegido por validación:", mejor.upper())

In [ ]:
def tabla(encabezado, filas):
    lineas = ["| " + " | ".join(encabezado) + " |", "|" + "|".join(["---"] * len(encabezado)) + "|"]
    lineas += ["| " + " | ".join(str(c) for c in f) + " |" for f in filas]
    return "\n".join(lineas)


def pct(x):
    return f"{100 * x:.1f} %"


r = resultados
md = ["# Resultados del entrenamiento", "", f"Generado el {datetime.now():%Y-%m-%d %H:%M}.", ""]
md += ["## 1. Datos", "",
       tabla(["Clase", "Entrenamiento", "Validación", "Prueba"],
             [[NOMBRES[i], conteos["ent"][i], conteos["val"][i], conteos["pru"][i]] for i in range(NUM_CLASES)]
             + [["**Total**", sum(conteos["ent"]), sum(conteos["val"]), sum(conteos["pru"])]]), ""]
md += ["## 2. Comparación MLP vs CNN (conjunto de prueba)", "",
       tabla(["Modelo", "Parámetros", "Exactitud", "Precisión (macro)", "Recall (macro)", "F1 (macro)",
              "Épocas", "Tiempo de entrenamiento", "Inferencia por imagen"],
             [[n.upper(), f"{r[n]['parametros']:,}".replace(",", " "), pct(r[n]["exactitud"]),
               pct(r[n]["precision_macro"]), pct(r[n]["recall_macro"]), pct(r[n]["f1_macro"]),
               r[n]["epocas"], f"{r[n]['segundos']:.0f} s", f"{r[n]['ms_por_imagen']:.2f} ms"]
              for n in ("mlp", "cnn")]), "",
       f"Modelo exportado a la demo: **{mejor.upper()}** (elegido por exactitud en validación: "
       f"MLP {pct(r['mlp']['val_exactitud'])}, CNN {pct(r['cnn']['val_exactitud'])}).", ""]
for k, n in (("3", "cnn"), ("4", "mlp")):
    md += [f"## {k}. Métricas por clase — {n.upper()}", "",
           tabla(["Clase", "Precisión", "Recall", "F1", "Soporte"],
                 [[c["clase"], pct(c["precision"]), pct(c["recall"]), pct(c["f1"]), c["soporte"]]
                  for c in r[n]["por_clase"]]), ""]
m = r["cnn"]["matriz"]
md += ["## 5. Matriz de confusión — CNN (filas = clase real, columnas = predicción)", "",
       tabla(["Real \\ Pred"] + ABREV, [[ABREV[i]] + m[i] for i in range(NUM_CLASES)]), "",
       "Abreviaturas: " + ", ".join(f"{a} = {n}" for a, n in zip(ABREV, NOMBRES)) + ".", "",
       "## 6. Pares más confundidos (CNN)", ""]
mm = np.array(m)
pares = sorted([(mm[i, j], i, j) for i in range(NUM_CLASES) for j in range(NUM_CLASES)
                if i != j and mm[i, j] > 0], reverse=True)[:6]
md += [tabla(["Clase real", "Predicha como", "Casos"], [[NOMBRES[i], NOMBRES[j], int(c)] for c, i, j in pares])
       if pares else "No hubo confusiones en el conjunto de prueba.", ""]
md += ["## 7. Configuración", "",
       tabla(["Parámetro", "Valor"], [
           ["Entrada", f"{TAM}x{TAM}x1, valores 0–1"],
           ["Optimizador", "Adam, lr inicial 0.001 (ReduceLROnPlateau x0.5)"],
           ["Pérdida", f"categorical_crossentropy con label smoothing {SUAVIZADO_ETIQUETAS}"],
           ["Épocas máximas", f"{EPOCAS} (EarlyStopping, paciencia 8)"],
           ["Tamaño de lote", LOTE],
           ["Aumento", "rotación ±9°, zoom ±12 %, desplazamiento ±8 %, ruido gaussiano σ=0.04"],
           ["Entrenamiento", f"demás integrantes (sin '{AUTOR_PRUEBA}'), ~82 % al azar" if AUTOR_PRUEBA else f"{round((1 - VAL - PRUEBA) * 100)} % al azar, estratificado por clase"],
           ["Validación", "~18 % al azar de los demás integrantes" if AUTOR_PRUEBA else f"{round(VAL * 100)} % al azar, estratificado por clase"],
           ["Prueba", f"TODO lo de '{AUTOR_PRUEBA}' (integrante no visto)" if AUTOR_PRUEBA else f"{round(PRUEBA * 100)} % al azar, estratificado por clase"],
           ["Unidad de división", "dibujo original (sus variantes aumentadas van juntas)"],
           ["Semilla", SEMILLA],
           ["TensorFlow / Keras", f"{tf.__version__} / {keras.__version__}"]]), ""]
texto = "\n".join(md)
with open(f"{SALIDA}/metricas.md", "w", encoding="utf-8") as f:
    f.write(texto)
display(Markdown(texto))

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
colores = {"mlp": "#9AA3B8", "cnn": "#2D4F9E"}
for n, h in historiales.items():
    ejes[0].plot(h["loss"], color=colores[n], ls="--", label=f"{n.upper()} entrenamiento")
    ejes[0].plot(h["val_loss"], color=colores[n], label=f"{n.upper()} validación")
    ejes[1].plot(h["accuracy"], color=colores[n], ls="--", label=f"{n.upper()} entrenamiento")
    ejes[1].plot(h["val_accuracy"], color=colores[n], label=f"{n.upper()} validación")
ejes[0].set_title("Pérdida (loss)")
ejes[1].set_title("Exactitud (accuracy)")
for e in ejes:
    e.set_xlabel("Época")
    e.grid(alpha=0.3)
    e.legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{SALIDA}/curvas.png", dpi=130)
plt.show()

for n in ("mlp", "cnn"):
    mat = np.array(resultados[n]["matriz"], dtype=float)
    norm = mat / np.maximum(mat.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLASES), ABREV)
    ax.set_yticks(range(NUM_CLASES), NOMBRES)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Clase real")
    ax.set_title(f"Matriz de confusión — {n.upper()} (prueba)")
    for i in range(NUM_CLASES):
        for j in range(NUM_CLASES):
            if mat[i, j]:
                ax.text(j, i, int(mat[i, j]), ha="center", va="center", fontsize=8,
                        color="white" if norm[i, j] > 0.5 else "#1A2238")
    fig.tight_layout()
    fig.savefig(f"{SALIDA}/matriz_{n}.png", dpi=130)
    plt.show()

pred, probs = resultados["cnn"]["pred"], resultados["cnn"]["probs"]
errores = np.where(pred != yp)[0][:24]
if len(errores):
    filas = int(np.ceil(len(errores) / 8))
    fig, ejes = plt.subplots(filas, 8, figsize=(12.8, filas * 1.9))
    for k, ax in enumerate(np.array(ejes).reshape(-1)):
        ax.axis("off")
        if k < len(errores):
            i = errores[k]
            ax.imshow(1 - Xp[i, :, :, 0], cmap="gray", vmin=0, vmax=1)
            ax.set_title(f"real {ABREV[yp[i]]}\npred {ABREV[pred[i]]} {probs[i].max():.0%}", fontsize=8)
    fig.tight_layout()
    fig.savefig(f"{SALIDA}/errores_cnn.png", dpi=120)
    plt.show()
else:
    print("La CNN no tuvo errores en la prueba.")

In [ ]:
modelo = modelos[mejor]
capas = []
for capa in modelo.layers:
    tipo, c = type(capa).__name__, capa.get_config()
    if tipo == "Conv2D":
        capas.append({"tipo": "conv2d", "filtros": c["filters"], "kernel": list(c["kernel_size"]),
                      "paso": list(c["strides"]), "padding": c["padding"], "activacion": c["activation"]})
    elif tipo == "MaxPooling2D":
        capas.append({"tipo": "maxpool", "tam": list(c["pool_size"]), "paso": list(c["strides"]),
                      "padding": c["padding"]})
    elif tipo == "Flatten":
        capas.append({"tipo": "flatten"})
    elif tipo == "Dense":
        capas.append({"tipo": "dense", "unidades": c["units"], "activacion": c["activation"]})
    elif tipo == "Dropout":
        capas.append({"tipo": "dropout"})

os.makedirs("modelo_web", exist_ok=True)
pesos, especificacion, inicio = [], [], 0
for w in modelo.get_weights():
    w = np.asarray(w, dtype="<f4")
    especificacion.append({"forma": list(w.shape), "inicio": inicio, "cantidad": int(w.size)})
    pesos.append(w.ravel())
    inicio += w.size
np.concatenate(pesos).tofile("modelo_web/pesos.bin")
info = {"arquitectura": mejor, "origen": "real", "fecha": f"{datetime.now():%Y-%m-%d}",
        "exactitud_prueba": round(resultados[mejor]["exactitud"], 4),
        "f1_macro_prueba": round(resultados[mejor]["f1_macro"], 4),
        "forma_entrada": [TAM, TAM, 1], "clases": CARPETAS, "capas": capas,
        "pesos": especificacion, "archivo_pesos": "pesos.bin"}
with open("modelo_web/modelo.json", "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=1)

limpio = {n: {k: v for k, v in res.items() if k not in ("pred", "probs")} for n, res in resultados.items()}
with open(f"{SALIDA}/metricas.json", "w", encoding="utf-8") as f:
    json.dump({"resultados": limpio, "historial": historiales, "conteos": conteos, "mejor": mejor},
              f, ensure_ascii=False, indent=1, default=float)

with zipfile.ZipFile("resultado_entrenamiento.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("modelo_web/modelo.json", "static/modelo/modelo.json")
    z.write("modelo_web/pesos.bin", "static/modelo/pesos.bin")
    for archivo in sorted(os.listdir(SALIDA)):
        if not archivo.endswith(".keras"):
            z.write(f"{SALIDA}/{archivo}", f"entrenamiento/resultados/{archivo}")
print("Modelo exportado:", mejor.upper(), "| exactitud prueba", info["exactitud_prueba"])

from google.colab import files
files.download("resultado_entrenamiento.zip")